# Перевод с русского на английский (GRU)

В Kaggle включи GPU и Internet, затем нажми Run All. Корпус скачивается при первом запуске. Результаты сохраняются в `/kaggle/working/translation_project/gru_simple/`. [Данные ManyThings](https://www.manythings.org/anki/)


In [ ]:
import io
import json
import random
import re
import subprocess
import sys
import time
import unicodedata
import urllib.request
import zipfile
from collections import Counter
from itertools import zip_longest
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence

try:
    from sacrebleu.metrics import BLEU, CHRF
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sacrebleu==2.5.1"])
    from sacrebleu.metrics import BLEU, CHRF

SEED = 42
VOCAB_SIZE = 5000
DIM = 128
BATCH_SIZE = 64
TOKEN_BUDGET = 2048
EPOCHS = 20
MAX_NEW_TOKENS = 128
DATA_PATH = None  # Локальный rus.txt или rus-eng.zip

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
OUT = Path("/kaggle/working/translation_project/gru_simple")
OUT.mkdir(parents=True, exist_ok=True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
DEVICE_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU"
print("Устройство", DEVICE_NAME)


## Данные

Все варианты перевода одной русской фразы попадают в одну часть выборки.


In [ ]:
URLS = [
    "https://www.manythings.org/anki/rus-eng.zip",
    "https://mail.manythings.org/anki/rus-eng.zip",
]
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
    "Accept": "application/zip, application/octet-stream;q=0.9, */*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.manythings.org/anki/",
}

def unzip(content):
    with zipfile.ZipFile(io.BytesIO(content)) as archive:
        return archive.read("rus.txt").decode("utf-8-sig")

def load_corpus():
    if DATA_PATH:
        path = Path(DATA_PATH)
        return unzip(path.read_bytes()) if path.suffix == ".zip" else path.read_text(encoding="utf-8-sig")
    path = OUT / "rus-eng.zip"
    if path.exists():
        try:
            return unzip(path.read_bytes())
        except (zipfile.BadZipFile, KeyError, UnicodeError):
            pass
    errors = []
    for url in URLS:
        try:
            request = urllib.request.Request(url, headers=HEADERS)
            with urllib.request.urlopen(request, timeout=45) as response:
                content = response.read()
            text = unzip(content)
        except (OSError, zipfile.BadZipFile, KeyError, UnicodeError) as error:
            errors.append(f"{url}: {error}")
            continue
        path.write_bytes(content)
        return text
    raise RuntimeError("Не удалось скачать корпус.\n" + "\n".join(errors))


In [ ]:
APOSTROPHES = str.maketrans("\u2019\u2018\u02bc", "'''")
TOKEN_PATTERN = re.compile(r"[^\W\d_]+(?:'[^\W\d_]+)*|\d+(?:[.,]\d+)*|[^\w\s]|_")

def tokenize(text):
    text = unicodedata.normalize("NFC", text).lower().translate(APOSTROPHES)
    return TOKEN_PATTERN.findall(text)

def detokenize(tokens):
    text = re.sub(r"\s+([.,!?;:%)\]}])", r"\1", " ".join(tokens))
    return re.sub(r"([(\[{])\s+", r"\1", text).strip()

text = load_corpus()
data = pd.DataFrame(
    [line.split("\t", 2) for line in text.splitlines() if line.count("\t") >= 2],
    columns=["en", "ru", "attribution"],
)
del text
for language in ["ru", "en"]:
    data[language] = data[language].map(lambda text: " ".join(tokenize(text)))
data = data[(data.ru != "") & (data.en != "")].drop_duplicates(["ru", "en"])

sources = sorted(data.ru.unique())
assert len(sources) >= 10, "Недостаточно корректных русских фраз."
random.Random(SEED + 1).shuffle(sources)
train_end = int(0.8 * len(sources))
val_end = train_end + int(0.1 * len(sources))
data["order"] = data.ru.map({source: i for i, source in enumerate(sources)})
data["split"] = "test"
data.loc[data.order < val_end, "split"] = "validation"
data.loc[data.order < train_end, "split"] = "train"
data = data.sort_values(["order", "en"]).drop(columns="order").reset_index(drop=True)

train = data[data.split == "train"]
val = data[data.split == "validation"]
test = data[data.split == "test"].groupby("ru", sort=False).agg(
    references=("en", list), attribution=("attribution", list)
).reset_index()
assert set(train.ru).isdisjoint(val.ru)
assert set(train.ru).isdisjoint(test.ru)
assert set(val.ru).isdisjoint(test.ru)
data.to_csv(OUT / "data.csv", index=False)
display(data.groupby("split").agg(pairs=("ru", "size"), sources=("ru", "nunique")))


## Словари и батчи

Словари строим по обучающей выборке. Длинные фразы собираем в меньшие батчи.


In [ ]:
SPECIAL = ["<pad>", "<unk>", "<bos>", "<eos>"]
PAD, UNK, BOS, EOS = range(4)

def build_vocab(texts):
    counts = Counter(token for text in texts for token in text.split())
    words = sorted(counts, key=lambda word: (-counts[word], word))
    return {word: i for i, word in enumerate(SPECIAL + words[:VOCAB_SIZE - 4])}

ru_vocab = build_vocab(train.ru)
en_vocab = build_vocab(train.en)
en_words = list(en_vocab)

def encode(text, vocab):
    ids = [BOS] + [vocab.get(token, UNK) for token in tokenize(text)] + [EOS]
    return torch.tensor(ids, dtype=torch.long)

def encode_pairs(sources, targets):
    return [
        (encode(ru, ru_vocab), encode(en, en_vocab))
        for ru, en in zip(sources, targets)
    ]

train_samples = encode_pairs(train.ru, train.en)
val_samples = encode_pairs(val.ru, val.en)
test_samples = encode_pairs(test.ru, (refs[0] for refs in test.references))


In [ ]:
def batch_indices(samples, shuffle=False, epoch=0):
    indices = list(range(len(samples)))
    if shuffle:
        random.Random(SEED + epoch).shuffle(indices)
    batch, longest = [], 0
    for index in indices:
        length = max(len(part) for part in samples[index])
        longest = max(longest, length)
        if batch and (len(batch) == BATCH_SIZE or (len(batch) + 1) * longest > TOKEN_BUDGET):
            yield batch
            batch, longest = [], length
        batch.append(index)
    if batch:
        yield batch

def batches(samples, shuffle=False, epoch=0):
    for indices in batch_indices(samples, shuffle, epoch):
        sources, targets = zip(*(samples[i] for i in indices))
        lengths = torch.tensor([len(source) for source in sources])
        yield (
            pad_sequence(sources, batch_first=True, padding_value=PAD).to(DEVICE),
            lengths,
            pad_sequence(targets, batch_first=True, padding_value=PAD).to(DEVICE),
        )


## Модель

Энкодер передаёт декодеру своё последнее скрытое состояние. Для энкодера учитываем длины фраз без PAD.


In [ ]:
class Seq2SeqGRU(nn.Module):
    def __init__(self):
        super().__init__()
        self.ru_embedding = nn.Embedding(len(ru_vocab), DIM, padding_idx=PAD)
        self.encoder = nn.GRU(DIM, DIM, batch_first=True)
        self.en_embedding = nn.Embedding(len(en_vocab), DIM, padding_idx=PAD)
        self.decoder = nn.GRU(DIM, DIM, batch_first=True)
        self.output = nn.Linear(DIM, len(en_vocab))
        self.dropout = nn.Dropout(0.1)

    def encode(self, source, lengths):
        embedded = self.dropout(self.ru_embedding(source))
        packed = pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        return self.encoder(packed)[1]

    def forward(self, source, lengths, target_input):
        hidden = self.encode(source, lengths)
        embedded = self.dropout(self.en_embedding(target_input))
        states, _ = self.decoder(embedded, hidden)
        return self.output(self.dropout(states))

torch.manual_seed(SEED)
model = Seq2SeqGRU().to(DEVICE)


## Обучение

Декодер учится предсказывать следующий токен по предыдущим токенам эталона. Для теста берём веса с минимальным loss на валидации.


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=PAD, reduction="sum")

def synchronize():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)

def run_epoch(samples, training=False, epoch=0):
    model.train(training)
    total_loss, tokens = 0.0, 0
    with torch.set_grad_enabled(training):
        for source, lengths, target in batches(samples, shuffle=training, epoch=epoch):
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(source, lengths, target[:, :-1])
            labels = target[:, 1:]
            loss = criterion(logits.reshape(-1, len(en_vocab)), labels.reshape(-1))
            count = labels.ne(PAD).sum()
            assert torch.isfinite(loss).item(), "Получен неконечный loss."
            if training:
                (loss / count).backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
                optimizer.step()
            total_loss += loss.item()
            tokens += count.item()
    return total_loss / tokens


In [ ]:
best_loss = float("inf")
bad_epochs = 0
history = []
synchronize()
training_start = time.perf_counter()

for epoch in range(EPOCHS):
    start = time.perf_counter()
    train_loss = run_epoch(train_samples, training=True, epoch=epoch)
    val_loss = run_epoch(val_samples)
    if val_loss < best_loss:
        best_loss = val_loss
        bad_epochs = 0
        torch.save({
            "model": model.state_dict(), "ru_vocab": ru_vocab, "en_vocab": en_vocab,
            "dim": DIM, "epoch": epoch + 1,
        }, OUT / "gru.pt")
    else:
        bad_epochs += 1
    synchronize()
    seconds = time.perf_counter() - start
    history.append([epoch + 1, train_loss, val_loss, seconds])
    print(f"Эпоха {epoch + 1}, train {train_loss:.3f}, val {val_loss:.3f}, {seconds:.1f} с")
    if epoch == 0:
        print(f"До лимита эпох около {seconds * (EPOCHS - 1) / 60:.1f} мин.")
    if bad_epochs == 3:
        break

training_seconds = time.perf_counter() - training_start
checkpoint = torch.load(OUT / "gru.pt", map_location=DEVICE, weights_only=True)
model.load_state_dict(checkpoint["model"])
history = pd.DataFrame(history, columns=["epoch", "train_loss", "val_loss", "seconds"])
history.to_csv(OUT / "history.csv", index=False)
print(f"Лучшая эпоха {checkpoint['epoch']}. Обучение заняло {training_seconds / 60:.2f} мин.")


In [ ]:
history.plot(x="epoch", y=["train_loss", "val_loss"], figsize=(7, 3))
plt.ylabel("Cross-entropy на токен")
plt.show()


## Генерация

На каждом шаге выбираем самый вероятный токен. Генерация заканчивается по EOS или после 128 токенов.


In [ ]:
def decode(ids):
    if EOS in ids:
        ids = ids[:ids.index(EOS)]
    return detokenize([en_words[i] for i in ids if i not in (PAD, BOS)])

@torch.inference_mode()
def generate(source, lengths):
    model.eval()
    hidden = model.encode(source, lengths)
    current = torch.full((source.size(0), 1), BOS, device=DEVICE, dtype=torch.long)
    output = torch.full((source.size(0), MAX_NEW_TOKENS), PAD, device=DEVICE, dtype=torch.long)
    finished = torch.zeros(source.size(0), dtype=torch.bool, device=DEVICE)
    for step in range(MAX_NEW_TOKENS):
        states, hidden = model.decoder(model.en_embedding(current), hidden)
        scores = model.output(states[:, -1])
        scores[:, [PAD, BOS]] = -torch.inf
        next_token = scores.argmax(dim=-1)
        next_token[finished] = PAD
        output[:, step] = next_token
        finished |= next_token.eq(EOS)
        current = next_token.unsqueeze(1)
        if finished.all().item():
            break
    return [decode(ids) for ids in output.cpu().tolist()]


## Тест

Для каждой русской фразы генерируем один перевод. BLEU и chrF учитывают все её эталоны.


In [ ]:
synchronize()
start = time.perf_counter()
predictions = []
for source, lengths, _ in batches(test_samples):
    predictions.extend(generate(source, lengths))
synchronize()
generation_seconds = time.perf_counter() - start
assert len(predictions) == len(test)

references = [[detokenize(text.split()) for text in refs] for refs in test.references]
reference_streams = list(zip_longest(*references, fillvalue=None))
bleu = BLEU(tokenize="13a")
chrf = CHRF(char_order=6, word_order=0, beta=2)
bleu_score = bleu.corpus_score(predictions, reference_streams).score
chrf_score = chrf.corpus_score(predictions, reference_streams).score

translations = test.assign(
    ru=test.ru.map(lambda text: detokenize(text.split())),
    prediction=predictions,
    reference=[refs[0] for refs in references],
)
display(translations[["ru", "reference", "prediction"]].sample(min(10, len(test)), random_state=SEED))
results = {
    "BLEU": bleu_score, "chrF": chrf_score, "best_epoch": checkpoint["epoch"],
    "train_pairs": len(train), "test_sources": len(test),
    "parameters": sum(p.numel() for p in model.parameters()),
    "training_seconds": training_seconds, "generation_seconds": generation_seconds,
    "seed": SEED, "dim": DIM, "vocab_limit": VOCAB_SIZE, "max_new_tokens": MAX_NEW_TOKENS,
    "batch_size": BATCH_SIZE, "token_budget": TOKEN_BUDGET, "learning_rate": 0.001,
    "epochs_completed": len(history), "device": DEVICE_NAME,
    "evaluation": "unique_ru/all_references",
    "bleu_signature": str(bleu.get_signature()), "chrf_signature": str(chrf.get_signature()),
}
print(f"BLEU {bleu_score:.2f}, chrF {chrf_score:.2f}")
print(f"Параметров {results['parameters']:,}. Перевод теста занял {generation_seconds:.2f} с.")
(OUT / "results.json").write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
for column in ["references", "attribution"]:
    translations[column] = translations[column].map(json.dumps)
translations.to_csv(OUT / "predictions.csv", index=False)


## Свои фразы


In [ ]:
def translate(text):
    source = encode(text, ru_vocab)
    if len(source) == 2:
        raise ValueError("Введите непустую фразу.")
    lengths = torch.tensor([len(source)])
    return generate(source.unsqueeze(0).to(DEVICE), lengths)[0]

for text in ["Я хочу пить.", "Где мой телефон?", "Она читает книгу."]:
    print(text, translate(text), sep="\n", end="\n\n")
